# Simulation RL Pipeline: Phantom Labels, NN Training, Multi-Agent Competition

This notebook implements the simulation-based RL pipeline:

1. **Part 1: Phantom Label Generation** — Create phantom-fill labels from simulation databases (output of `simulation_stylized_facts.ipynb`)
2. **Part 2: Neural Network Training** — Train a fill-probability MLP on simulation labels
3. **Part 3: Multi-Agent RL Competition** — Run multi-agent market-maker competitions with RL-based fill probability updates

In [ ]:
%pip install -e ..

from __future__ import annotations

import json
import random
import re
import shutil
import sqlite3
import time
from collections import deque
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from joblib import Parallel, delayed

try:
    import pyarrow  # noqa: F401
except ImportError as e:
    raise ImportError(
        "Install pyarrow to write parquet (e.g. `%pip install pyarrow`)."
    ) from e

from research_core.classes import (
    PhantomLabelConfig,
    PhantomLabeller,
    SimulateFast,
    AnalyseMarket,
    helpers,
    resolve_data_path,
    write_day_parquet,
    write_feature_schema,
    write_manifest,
)
from research_core.classes import mm_backtest_parallel
from research_core.classes.mm_competition import run_competition_sim
from research_core.classes.fill_rl import (
    run_fill_rl_sim,
    run_fill_rl_multi_sim,
    update_fill_nn,
    calibration_table,
    depth_calibration_table,
    bce_score,
    brier_score,
    delta_ticks_from_X,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
sim_db_dir = helpers.project_root() / "data"
sim_db_pattern = "sim_events_1M_*.sqlite"
sim_dbs = sorted(sim_db_dir.glob(sim_db_pattern))
assert len(sim_dbs) > 0, (
    f"No simulation databases found matching {sim_db_dir / sim_db_pattern}. "
    "Run simulation_stylized_facts.ipynb first."
)
print(f"Found {len(sim_dbs)} simulation databases")
for db in sim_dbs[:5]:
    print(f"  {db.name}")
if len(sim_dbs) > 5:
    print(f"  ... and {len(sim_dbs) - 5} more")


In [ ]:
# --- Source: a BATCH of whole-LO SimulateFast runs (recording_mode='full') ---
# sim_dbs loaded in previous cell (glob over data/sim_events_1M_*.sqlite)
n_runs = len(sim_dbs)

# Each run -> this many contiguous time chunks (pseudo-days).  Runs are already
# independent so 1 is simplest; raise it only to cap per-day rows/memory if a
# single ~10M-event run is too large for the labeller to load at once.
n_chunks_per_run = 1

symbol = "SIM"

tick_size = 0.05
tau = 1.0
delta_lo = -0.50

# Phantom order size (shares).  The fill label is an atomic full-fill —
# ``mo_volume >= queue_ahead + phantom_size`` — so it must match the size the
# MM agents actually post in the competition runs.  A 1000-lot order needs a
# far larger MO to fully execute than the legacy size-1 phantom, so the
# fill-rate-vs-depth curve (and hence the optimal quoting depth) shifts with
# size.  Set this to the deployed agent order size (see ``size`` in the
# mm_competition sweep / simulation.ipynb).
phantom_size = 10000

# Phantom-label output — separate directory so existing labels are never overwritten.
phantom_out_rel = f"phantom_labels_sim_1M_size{phantom_size}"
phantom_out_dir = resolve_data_path(phantom_out_rel)


# Match phantom_training_data.ipynb EXACTLY so only the *data* differs.  The
# empirical pipeline labels the full δ ∈ [-0.50, 2.00] PLN grid (n_delta=51),
# with n_queue_levels=40 and n_book_levels=0.  We reuse that config verbatim;
# the sim DB only records a few depth levels, so PhantomLabeller zero-fills the
# missing bid/ask_depth_L* columns and queue_ahead saturates for deep δ.
max_delta = 2.0              # empirical: max_delta
n_queue_levels = 40          # empirical: n_queue_levels (missing sim levels zero-filled)
n_book_levels_target = 0     # empirical: n_book_levels (queue_ahead suffices)

# recent_vol feature definition.  The simulated order flow runs at a far higher
# event rate than the empirical session, so the legacy per-event EWMA vol is on
# a wildly different scale than anything reconstructible at backtest inference.
# Use the cadence-independent trailing realised vol on a uniform time grid:
# RMS of log-mid returns over the last vol_window_s seconds, sampled every
# vol_sample_dt_s seconds (200 x 1 s by default).  This is the SAME definition
# the NN must be fed at inference (mm_backtest_parallel).
vol_mode = "realized_time"
vol_window_s = 200.0
vol_sample_dt_s = 1.0


def max_depth_level(db: Path, side: str) -> int:
    """Number of ``{side}_depth_L*`` columns present in the sim ``orders`` table."""
    if not Path(db).is_file():
        return 0
    with sqlite3.connect(str(db)) as c:
        cols = [r[1] for r in c.execute("PRAGMA table_info(orders)").fetchall()]
    levels = [int(m.group(1)) for col in cols
          for m in [re.match(rf"{side}_depth_L(\d+)$", col)] if m]
    return (max(levels) + 1) if levels else 0


detected_levels = min(max_depth_level(sim_dbs[0], "bid"), max_depth_level(sim_dbs[0], "ask"))

print(f"Source runs: {len(sim_dbs)} DBs (1M family, n_runs={n_runs}, "
      f"n_chunks_per_run={n_chunks_per_run})")
print(f"Delta grid: [{delta_lo}, {max_delta}] PLN  step={tick_size}  (matches empirical)")
print(f"Phantom size: {phantom_size} shares (atomic full-fill label; match to MM agent order size)")
print(f"Hawkes filter: single-kernel default (kghm_multivariate_single)")
print(f"Recorded sim depth levels (min of bid/ask): {detected_levels}; "
      f"n_queue_levels={n_queue_levels} (missing levels zero-filled)")
if detected_levels and max_delta > detected_levels * tick_size + 1e-9:
    print(f"  NOTE: max_delta {max_delta} PLN exceeds recorded depth "
          f"({detected_levels} levels = {detected_levels * tick_size} PLN); "
          f"queue_ahead saturates for deep deltas (labels only; grid/schema unchanged).")

phantom_cfg = PhantomLabelConfig(
    tau=tau,
    cadence="bbo_change",
    delta_lo=delta_lo,
    max_delta=max_delta,
    tick_size=tick_size,
    phantom_size=phantom_size,
    vol_mode=vol_mode,
    vol_window_s=vol_window_s,
    vol_sample_dt_s=vol_sample_dt_s,
    vol_halflife=50,
    n_book_levels=n_book_levels_target,
    n_queue_levels=n_queue_levels,
)

In [ ]:
# Stage A — per-run adapter.  Each source run is adapted INTO ITS OWN small DB
# in the empirical schema (one combined ~60×10M-row DB would be unwieldy), so
# the labeller processes one ~10M-event run at a time — exactly like the
# original single-DB workflow.  `run_days` are the (globally unique) pseudo-day
# keys for this run; the run is cut into len(run_days) contiguous time chunks.
#
# Streamed in chunks so a 10M-event 'full' DB never loads fully into memory.
# Rows are already in time order, so no global sort is needed (the labeller
# re-sorts per day).  `timestamp` is stored as int64 nanoseconds: the labeller
# derives t_ns via `pd.to_datetime(timestamp).astype("int64")`, round-tripping
# int-ns exactly (datetime *strings* would lose sub-second precision).

_existing_parquets = list(phantom_out_dir.glob(f"{symbol}_*.parquet")) if phantom_out_dir.is_dir() else []
_skip_labelling = len(_existing_parquets) > 0

if _skip_labelling:
    print(f"Part 1 CACHED: {len(_existing_parquets)} parquet(s) already exist in {phantom_out_dir.name}/")
    print("  Delete the directory to regenerate labels.")


def adapt_one_run(sim_db, out_db, run_days, read_chunksize=500_000):
    sim_db, out_db = Path(sim_db), Path(out_db)
    if not sim_db.is_file():
        raise FileNotFoundError(f"Simulation DB not found: {sim_db}")
    for sidecar in (out_db, Path(str(out_db) + "-wal"), Path(str(out_db) + "-shm")):
        if sidecar.is_file():
            sidecar.unlink()

    src = sqlite3.connect(str(sim_db))
    try:
        tmin, tmax = src.execute(
            "SELECT MIN(timestamp), MAX(timestamp) FROM orders"
        ).fetchone()
        if tmin is None:
            raise ValueError(f"sim `orders` table is empty: {sim_db.name}")
        edges = np.linspace(float(tmin), float(tmax), len(run_days) + 1)
        inner = edges[1:-1]   # interior edges -> chunk index via searchsorted

        def _days(ts: np.ndarray) -> List[str]:
            idx = np.searchsorted(inner, ts, side="right")
            return [run_days[int(i)] for i in idx]

        # Copy only the columns PhantomLabeller reads (the orders table has
        # ~40 cols; the labeller uses ~21). Big saving on a 10M-row copy.
        src_ocols = [r[1] for r in src.execute("PRAGMA table_info(orders)")]
        need_o = ["timestamp", "event_type", "order_id", "side", "order_price",
                  "volume", "best_bid", "best_ask", "mid_price", "spread_ticks",
                  "imbalance"]
        depth_o = [c for c in src_ocols if re.match(r"(bid|ask)_depth_L\d+$", c)]
        sel_o = [c for c in need_o if c in src_ocols] + depth_o
        q_orders = "SELECT " + ", ".join(sel_o) + " FROM orders"

        src_mcols = [r[1] for r in src.execute("PRAGMA table_info(mo_orders)")]
        sel_m = [c for c in ["timestamp", "side", "mo_volume"] if c in src_mcols]
        q_mos = "SELECT " + ", ".join(sel_m) + " FROM mo_orders"

        dst = sqlite3.connect(str(out_db))
        n_o = n_m = 0
        try:
            dst.execute("PRAGMA journal_mode=WAL")
            dst.execute("PRAGMA synchronous=NORMAL")

            first = True
            for ch in pd.read_sql_query(q_orders, src, chunksize=read_chunksize):
                ts = ch["timestamp"].to_numpy(dtype=np.float64)
                ch["day"] = _days(ts)
                ch["timestamp"] = (ts * 1e9).astype(np.int64)
                ch.to_sql("orders", dst,
                          if_exists="replace" if first else "append", index=False)
                n_o += len(ch); first = False

            first = True
            for ch in pd.read_sql_query(q_mos, src, chunksize=read_chunksize):
                ts = ch["timestamp"].to_numpy(dtype=np.float64)
                ch["day"] = _days(ts)
                ch["first_time_ns"] = (ts * 1e9).astype(np.int64)
                ch.to_sql("mo_orders", dst,
                          if_exists="replace" if first else "append", index=False)
                n_m += len(ch); first = False

            dst.execute("CREATE INDEX IF NOT EXISTS idx_orders_day_ts ON orders(day, timestamp)")
            dst.execute("CREATE INDEX IF NOT EXISTS idx_mo_day_ts ON mo_orders(day, first_time_ns)")
            dst.commit()
        finally:
            dst.close()
    finally:
        src.close()
    return n_o, n_m

# Stage B — for each source run: adapt -> label its day(s) -> write parquet.
# One small temp DB is reused per run (kept on disk only briefly), so peak disk
# use stays at ~one run rather than the whole 60-run batch.

if not _skip_labelling:
    phantom_out_dir.mkdir(parents=True, exist_ok=True)
    stale_files = list(phantom_out_dir.glob(f"{symbol}_*.parquet"))
    for stale_path in stale_files:
        stale_path.unlink()
    if stale_files:
        print(f"Cleared {len(stale_files)} stale parquet(s) in {phantom_out_dir.name}/")
    tmp_db = resolve_data_path("_phantom_run_tmp_1M.sqlite")

    successful_days: List[str] = []
    failed_days: List[tuple[str, str]] = []
    day_sources: dict = {}
    total_rows = 0
    day_counter = 0
    n_kernels: Optional[int] = None
    feature_schema_dict: Optional[dict] = None
    t0 = time.time()

    for run_idx, sim_db in enumerate(sim_dbs):
        run_days = [f"sim{day_counter + k:05d}" for k in range(n_chunks_per_run)]
        day_counter += n_chunks_per_run
        try:
            adapt_one_run(sim_db, tmp_db, run_days)
        except Exception as exc:
            print(f"  [{run_idx + 1}/{len(sim_dbs)}] {sim_db.name}: ADAPT FAILED — {exc!r}",
                  flush=True)
            for day in run_days:
                failed_days.append((day, f"adapt: {exc!r}"))
            continue

        labeller = PhantomLabeller(
            tmp_db, config=phantom_cfg, hawkes=True,
        )
        if feature_schema_dict is None:
            feature_schema_dict = labeller.feature_schema()
            n_kernels = labeller._n_kernels
            print(f"features per t: {len(labeller.feature_names)} | "
                  f"n_kernels={n_kernels} | n_delta={labeller.n_delta}", flush=True)

        for day_key in labeller.list_days():
            t_day = time.time()
            try:
                df = labeller.label_day(day_key)
                elapsed = time.time() - t_day
                if df.empty:
                    print(f"  {day_key} ({sim_db.name}): empty [{elapsed:.1f}s]", flush=True)
                    continue
                out_path = phantom_out_dir / f"{symbol}_{day_key}.parquet"
                n_rows = write_day_parquet(df, out_path)
                successful_days.append(day_key)
                day_sources[day_key] = sim_db.name
                total_rows += n_rows
                print(f"  [{run_idx + 1}/{len(sim_dbs)}] {day_key} ({sim_db.name}): "
                      f"{n_rows:,} rows [{elapsed:.1f}s]", flush=True)
            except Exception as exc:
                err = repr(exc)
                failed_days.append((day_key, err))
                print(f"  {day_key} ({sim_db.name}): FAILED — {err.splitlines()[0]}",
                      flush=True)

    for sidecar in (tmp_db, Path(str(tmp_db) + "-wal"), Path(str(tmp_db) + "-shm")):
        if sidecar.is_file():
            sidecar.unlink()

    runtime = time.time() - t0
    print(
        f"\nDone in {runtime:.1f}s | runs={len(sim_dbs)} "
        f"success_days={len(successful_days)} failed={len(failed_days)} "
        f"total_rows={total_rows:,}"
    )

    if successful_days:
        write_feature_schema(feature_schema_dict, phantom_out_dir)
        write_manifest(
            phantom_out_dir,
            db_path=f"{len(sim_dbs)} runs: 1M x range({n_runs})",
            days_processed=successful_days,
            config=phantom_cfg,
            total_rows=total_rows,
            runtime_seconds=runtime,
            git_commit=None,
            hawkes_n_kernels=n_kernels,
        )
        print(f"Wrote feature_schema.json + manifest.json -> {phantom_out_dir}")
    if failed_days:
        print("Failed (first line only):")
        for day, err in failed_days[:20]:
            print(f"  {day}: {err.splitlines()[0]}")

## Part 2: NN Training (Simulation Data)


In [ ]:
# Labels from phantom_training_data_sim copy.ipynb (sim_events_1M_{i}.sqlite).
phantom_size = 10000
symbol = "SIM"
phantom_dir = resolve_data_path(f"phantom_labels_sim_1M_size{phantom_size}")
max_rows_per_day: Optional[int] = None
batch_size = 65536
epochs = 40
lr = 3e-4
hidden = [64, 32]
dropout = 0.15
seed = 42
pos_weight: Optional[float] = None  # e.g. 2.0 if fills are rare

# queue_ahead feature transform.  "log1p" applies np.log1p before
# normalisation so the model has better resolution near BBO where
# queue_ahead is small.  None keeps the raw value (legacy behaviour).
# Recorded in the checkpoint so inference auto-selects the matching
# transform.
queue_transform: Optional[str] = "log1p"

qt_suffix = "_logq" if queue_transform == "log1p" else ""
out_dir = resolve_data_path(f"phantom_models_sim_1M_size{phantom_size}{qt_suffix}")

_ckpt_exists = (out_dir / "fillprob_mlp_best.pt").exists()
if _ckpt_exists:
    print(f"NOTE: Checkpoint already exists at {out_dir / 'fillprob_mlp_best.pt'}")
    print("  Training will overwrite it.")

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
schema_path = phantom_dir / "feature_schema.json"
manifest_path = phantom_dir / "manifest.json"
assert schema_path.exists(), f"Missing {schema_path}"
assert manifest_path.exists(), f"Missing {manifest_path}"

with open(schema_path) as f:
    schema = json.load(f)
with open(manifest_path) as f:
    manifest = json.load(f)

# recent_vol training definition (recorded into the checkpoint so the backtest
# reconstructs the SAME feature at inference; see mm_backtest_parallel auto mode).
manifest_cfg = manifest.get("config", {})
label_phantom_size = int(manifest_cfg.get("phantom_size", schema.get("phantom_size", -1)))
assert label_phantom_size == phantom_size, (
    f"Label phantom_size={label_phantom_size} != config phantom_size={phantom_size}. "
    "Re-run phantom_training_data_sim copy.ipynb or fix phantom_size."
)
vol_mode = manifest_cfg.get("vol_mode", "ewma_event")
vol_window_s = manifest_cfg.get("vol_window_s", None)
vol_sample_dt_s = manifest_cfg.get("vol_sample_dt_s", None)
assert vol_mode == "realized_time", (
    f"Expected realized_time labels, got vol_mode={vol_mode!r}. "
    "Re-run phantom_training_data_sim copy.ipynb."
)
print(f"recent_vol mode: {vol_mode}"
      + (f" (window={vol_window_s}s, dt={vol_sample_dt_s}s)"
         if vol_mode == "realized_time" else ""))

tick_size = schema["tick_size"]
tau = schema["tau"]
n_delta = schema["delta_grid"]["n_points"]

feat_cols = [c for c, info in schema["columns"].items() if info["role"] == "feature"]
input_cols_schema = [c for c, info in schema["columns"].items() if info["role"] == "input"]
print(f"Feature columns ({len(feat_cols)}): {feat_cols}")
print(f"Input columns: {input_cols_schema}")
print(f"tick_size={tick_size}, tau={tau}, n_delta={n_delta}")

all_days = manifest["days_processed"]
print(f"Total days/chunks in manifest: {len(all_days)}")

print(f"Parquet prefix (symbol): {symbol!r}")

# Chronological day/chunk split (NO shuffling across days => no temporal
# leakage).  Fractions adapt to however many chunks the sim produced.
train_frac, val_frac = 0.80, 0.10
n_days = len(all_days)
assert n_days >= 3, f"Need >=3 days/chunks for a train/val/test split, got {n_days}"
n_train = min(max(1, round(n_days * train_frac)), n_days - 2)
n_val = min(max(1, round(n_days * val_frac)), n_days - n_train - 1)

train_days = all_days[:n_train]
val_days = all_days[n_train : n_train + n_val]
test_days = all_days[n_train + n_val :]

print(f"\nSplit: train={len(train_days)}, val={len(val_days)}, test={len(test_days)}")
print(f"  Train: {train_days[0]} .. {train_days[-1]}")
print(f"  Val:   {val_days[0]} .. {val_days[-1]}")
print(f"  Test:  {test_days[0]} .. {test_days[-1]}")

def day_to_path(day_key: str) -> Path:
    return phantom_dir / f"{symbol}_{day_key}.parquet"

train_files = [day_to_path(day_key) for day_key in train_days]
val_files = [day_to_path(day_key) for day_key in val_days]
test_files = [day_to_path(day_key) for day_key in test_days]

for fpath in train_files + val_files + test_files:
    assert fpath.exists(), f"Missing parquet: {fpath}"

In [ ]:
norm_cols = feat_cols + ["delta", "queue_ahead"]
n_norm = len(norm_cols)
qa_idx = norm_cols.index("queue_ahead")

running_sum = np.zeros(n_norm, dtype=np.float64)
running_sq = np.zeros(n_norm, dtype=np.float64)
running_n = 0

t0 = time.time()
for fi, fpath in enumerate(train_files):
    df = pd.read_parquet(fpath, columns=norm_cols)
    if max_rows_per_day is not None and len(df) > max_rows_per_day:
        df = df.sample(n=max_rows_per_day, random_state=seed)
    vals = df[norm_cols].values.astype(np.float64)
    if queue_transform == "log1p":
        vals[:, qa_idx] = np.log1p(vals[:, qa_idx])
    running_sum += vals.sum(axis=0)
    running_sq += (vals ** 2).sum(axis=0)
    running_n += len(vals)
    if (fi + 1) % max(1, len(train_files) // 5) == 0:
        print(f"  Stats pass: {fi+1}/{len(train_files)} files, {running_n:,} rows")

feat_mean = (running_sum / running_n).astype(np.float32)
feat_std = np.sqrt(running_sq / running_n - feat_mean.astype(np.float64) ** 2).astype(np.float32)
feat_std[feat_std < 1e-8] = 1.0

print(f"\nFeature stats computed in {time.time() - t0:.1f}s over {running_n:,} rows")
print(f"  queue_transform: {queue_transform or 'None (raw)'}")
for i, col in enumerate(norm_cols):
    print(f"  {col:30s}  mean={feat_mean[i]:+10.4f}  std={feat_std[i]:10.4f}")

In [ ]:
def load_day_tensors(
    path: Path,
    feat_cols: List[str],
    feat_mean: np.ndarray,
    feat_std: np.ndarray,
    max_rows: Optional[int] = None,
    queue_transform: Optional[str] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Load one day's parquet, normalize, shuffle, return (X, y) tensors."""
    df = pd.read_parquet(path)
    if max_rows is not None and len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=None)
    df = df.sample(frac=1.0)

    norm_cols = feat_cols + ["delta", "queue_ahead"]
    norm_vals = df[norm_cols].values.astype(np.float32)
    if queue_transform == "log1p":
        qa_idx = norm_cols.index("queue_ahead")
        norm_vals[:, qa_idx] = np.log1p(norm_vals[:, qa_idx])
    norm_vals = (norm_vals - feat_mean) / feat_std

    side = (df["side"].values.astype(np.float32) - 1.0).reshape(-1, 1)

    X = np.hstack([norm_vals, side])
    y = df["label"].values.astype(np.float32)

    return torch.from_numpy(X), torch.from_numpy(y)

In [ ]:
class FillProbMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int], dropout: float = 0.0):
        super().__init__()
        layers: list = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


in_dim = len(feat_cols) + 3  # features + delta + queue_ahead + side
model = FillProbMLP(in_dim, hidden, dropout=dropout).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
pw = torch.tensor([pos_weight], device=device) if pos_weight else None
criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

train_losses: list[float] = []
val_losses: list[float] = []
val_briers: list[float] = []
best_val_loss = float("inf")
best_state: Optional[dict] = None

t_start = time.time()
for epoch in range(epochs):
    t_epoch = time.time()

    # --- Train ---
    model.train()
    epoch_loss, epoch_n = 0.0, 0
    for fpath in train_files:
        X, y = load_day_tensors(fpath, feat_cols, feat_mean, feat_std, max_rows_per_day, queue_transform=queue_transform)
        X, y = X.to(device), y.to(device)
        for i in range(0, len(X), batch_size):
            xb, yb = X[i : i + batch_size], y[i : i + batch_size]
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
            epoch_n += len(xb)
    train_losses.append(epoch_loss / epoch_n)

    # --- Val ---
    model.eval()
    val_loss_sum, val_brier_sum, val_n = 0.0, 0.0, 0
    with torch.no_grad():
        for fpath in val_files:
            X, y = load_day_tensors(fpath, feat_cols, feat_mean, feat_std, None, queue_transform=queue_transform)
            X, y = X.to(device), y.to(device)
            logits = model(X)
            val_loss_sum += nn.functional.binary_cross_entropy_with_logits(
                logits, y, reduction="sum"
            ).item()
            probs = torch.sigmoid(logits)
            val_brier_sum += ((probs - y) ** 2).sum().item()
            val_n += len(y)
    vl = val_loss_sum / val_n
    vb = val_brier_sum / val_n
    val_losses.append(vl)
    val_briers.append(vb)

    if vl < best_val_loss:
        best_val_loss = vl
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| train_loss={train_losses[-1]:.5f} "
        f"| val_loss={vl:.5f} "
        f"| val_brier={vb:.5f} "
        f"| lr={scheduler.get_last_lr()[0]:.2e} "
        f"| {time.time() - t_epoch:.1f}s"
    )

print(f"\nBest val loss: {best_val_loss:.5f}  (total {time.time() - t_start:.0f}s)")

In [ ]:
# --- Restore best checkpoint ---
assert best_state is not None, "No best state saved — training may have failed"
model.load_state_dict(best_state)
model.eval()

# --- Collect test predictions ---
all_probs, all_labels, all_delta, all_queue, all_side = [], [], [], [], []

with torch.no_grad():
    for fpath in test_files:
        df = pd.read_parquet(fpath)
        norm_cols_local = feat_cols + ["delta", "queue_ahead"]
        norm_vals = df[norm_cols_local].values.astype(np.float32)
        if queue_transform == "log1p":
            qa_idx = norm_cols_local.index("queue_ahead")
            norm_vals[:, qa_idx] = np.log1p(norm_vals[:, qa_idx])
        norm_vals = (norm_vals - feat_mean) / feat_std
        side_enc = (df["side"].values.astype(np.float32) - 1.0).reshape(-1, 1)
        X_t = torch.from_numpy(np.hstack([norm_vals, side_enc])).to(device)
        logits = model(X_t)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(df["label"].values.astype(np.float32))
        all_delta.append(df["delta"].values.astype(np.float32))
        all_queue.append(df["queue_ahead"].values.astype(np.float32))
        all_side.append(df["side"].values.astype(np.int8))

probs_test = np.concatenate(all_probs)
labels_test = np.concatenate(all_labels)
delta_test = np.concatenate(all_delta)
queue_test = np.concatenate(all_queue)
side_test = np.concatenate(all_side)

# --- NN test metrics ---
eps = 1e-7
nn_bce = -np.mean(
    labels_test * np.log(probs_test + eps)
    + (1 - labels_test) * np.log(1 - probs_test + eps)
)
nn_brier = np.mean((probs_test - labels_test) ** 2)

# --- Constant baseline: predict mean(y_train) ---
train_mean_y = 0.0
train_total = 0
for fpath in train_files:
    df = pd.read_parquet(fpath, columns=["label"])
    if max_rows_per_day is not None and len(df) > max_rows_per_day:
        df = df.sample(n=max_rows_per_day, random_state=seed)
    train_mean_y += df["label"].sum()
    train_total += len(df)
train_mean_y /= train_total

const_bce = -np.mean(
    labels_test * np.log(train_mean_y + eps)
    + (1 - labels_test) * np.log(1 - train_mean_y + eps)
)
const_brier = np.mean((train_mean_y - labels_test) ** 2)

# --- Queue-only linear baseline ---
lin_X_parts, lin_y_parts = [], []
for fpath in train_files:
    df = pd.read_parquet(fpath, columns=["delta", "queue_ahead", "side", "label"])
    if max_rows_per_day is not None and len(df) > max_rows_per_day:
        df = df.sample(n=max_rows_per_day, random_state=seed)
    x_lin = np.column_stack([
        df["delta"].values.astype(np.float32),
        df["queue_ahead"].values.astype(np.float32),
        (df["side"].values.astype(np.float32) - 1.0),
    ])
    lin_X_parts.append(x_lin)
    lin_y_parts.append(df["label"].values.astype(np.float32))

lin_X_train = torch.from_numpy(np.concatenate(lin_X_parts)).to(device)
lin_y_train = torch.from_numpy(np.concatenate(lin_y_parts)).to(device)

lin_model = nn.Linear(3, 1).to(device)
lin_opt = torch.optim.Adam(lin_model.parameters(), lr=1e-2)
lin_crit = nn.BCEWithLogitsLoss()

for _ in range(50):
    for i in range(0, len(lin_X_train), batch_size):
        xb = lin_X_train[i : i + batch_size]
        yb = lin_y_train[i : i + batch_size]
        logit = lin_model(xb).squeeze(-1)
        loss = lin_crit(logit, yb)
        lin_opt.zero_grad()
        loss.backward()
        lin_opt.step()

lin_X_test = torch.from_numpy(
    np.column_stack([delta_test, queue_test, side_test.astype(np.float32) - 1.0])
).to(device)

with torch.no_grad():
    lin_probs = torch.sigmoid(lin_model(lin_X_test).squeeze(-1)).cpu().numpy()

lin_bce = -np.mean(
    labels_test * np.log(lin_probs + eps)
    + (1 - labels_test) * np.log(1 - lin_probs + eps)
)
lin_brier = np.mean((lin_probs - labels_test) ** 2)

print(f"{'Model':<20s} {'BCE':>10s} {'Brier':>10s}")
print("-" * 42)
print(f"{'Constant baseline':<20s} {const_bce:10.5f} {const_brier:10.5f}")
print(f"{'Queue-only linear':<20s} {lin_bce:10.5f} {lin_brier:10.5f}")
print(f"{'FillProbMLP':<20s} {nn_bce:10.5f} {nn_brier:10.5f}")
print(f"\nTrain fill rate: {train_mean_y:.4f}")
print(f"Test  fill rate: {labels_test.mean():.4f}")
print(f"Test  n_rows:    {len(labels_test):,}")


out_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = out_dir / "fillprob_mlp_best.pt"
torch.save(
    {
        "state_dict": best_state,
        "feat_mean": feat_mean,
        "feat_std": feat_std,
        "feat_cols": feat_cols,
        "hidden": hidden,
        "dropout": dropout,
        "in_dim": in_dim,
        "tau": tau,
        "tick_size": tick_size,
        "vol_mode": vol_mode,
        "vol_window_s": vol_window_s,
        "vol_sample_dt_s": vol_sample_dt_s,
        "queue_transform": queue_transform,
    },
    ckpt_path,
)
print(f"Saved checkpoint: {ckpt_path}")

config_path = out_dir / "train_config.json"
config = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "phantom_dir": str(phantom_dir),
    "symbol": symbol,
    "seed": seed,
    "epochs": epochs,
    "lr": lr,
    "hidden": hidden,
    "dropout": dropout,
    "batch_size": batch_size,
    "max_rows_per_day": max_rows_per_day,
    "pos_weight": pos_weight,
    "queue_transform": queue_transform,
    "train_days": train_days,
    "val_days": val_days,
    "test_days": test_days,
    "best_val_loss": best_val_loss,
    "test_bce": float(nn_bce),
    "test_brier": float(nn_brier),
    "const_bce": float(const_bce),
    "queue_only_bce": float(lin_bce),
    "train_fill_rate": float(train_mean_y),
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved config:     {config_path}")

## Part 3: Multi-Agent RL Competition


In [ ]:
t_events     = 1_000_000
arrival_mode = "hawkes_multivariate"
# 0.10 zeroes the episode mid drift (ovn2/ovn3 suites, Jul 2026):
# agent-free drift map crosses zero at eps~0.10 and live gamma=0.1 runs
# show +9.0 +/- 4.7 ticks/ep vs -13.4 +/- 4.4 at the old 0.05.
drift_eps    = 0.10

resil_kappa      = 0.0817
resil_phi        = 0.0746
resil_tau_s      = 4.5
resil_flow_tau_s = 32.3

nn_sim_ckpt = str(resolve_data_path("phantom_models_sim_1M_size10000_logq/fillprob_mlp_best.pt"))
sim_tick    = 0.05
gamma       = 0.1
size        = 10000
t_comp      = 200_000
base_sims_per_setup = 12
fair_ref_n_agents   = 10
data_scale_global   = 2
n_workers   = 12
base_seed   = 20240604

snapshot_kwargs = dict(asset="KGHM", day_key="d20170110", snapshot_time="10:00:00", tick_size=sim_tick)

erg_params_sim = {"k": 16.01693035489198, "vol_halflife": 200,
                  "tick_size": sim_tick, "intensity_a": 0.05313567421660872}

out_dir = Path(nn_sim_ckpt).resolve().parents[1] / "mm_competition"
out_dir.mkdir(parents=True, exist_ok=True)


def fair_n_sims(n_agents, base=base_sims_per_setup):
    return max(1, int(round(base * (fair_ref_n_agents / n_agents) * data_scale_global)))


comp_kwargs = dict(
    T=t_comp, ckpt_path=nn_sim_ckpt, snapshot_kwargs=snapshot_kwargs,
    erg_params=erg_params_sim, size=size, gamma=gamma,
    solver_tick=sim_tick / 2, max_delta=2.0,
    poisson_tau=1.0, delta_lo=0.0, max_iter=1000, tol=1e-3,
    arrival_mode=arrival_mode, drift_eps=drift_eps,
    requote_cadence=1.0, base_seed=base_seed,
    day_span_s=None, vol_feature_mode="auto",
    agents_affect_mo_sizing=False, lo_inside_spread_scale=0.3,
    solver_engine="hull",
    resil_kappa=resil_kappa, resil_tau_s=resil_tau_s,
    resil_phi=resil_phi, resil_flow_tau_s=resil_flow_tau_s,
    out_dir=str(out_dir),
)


def save_setup(results, n_agents, out_dir):
    setup_dir = out_dir / f"setup_{n_agents:02d}"
    setup_dir.mkdir(parents=True, exist_ok=True)
    rows = [row for run in results for row in run["summary"]]
    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary.to_parquet(setup_dir / "summary.parquet", index=False)
    return summary


def run_setup(n_agents, kwargs, n_sims=None, workers=n_workers):
    if n_sims is None:
        n_sims = fair_n_sims(n_agents)
    t0 = time.time()
    results = Parallel(n_jobs=min(workers, n_sims), verbose=10)(
        delayed(run_competition_sim)(rid, n_agents, **kwargs)
        for rid in range(n_sims)
    )
    print(f"Setup {n_agents}: {n_sims} sims in {(time.time()-t0)/60:.1f} min")
    return results

def plot_final_price_histogram(results, bins=20):
    if isinstance(results, dict):
        results = [results]
    price_changes = np.array([float(r['fills_price'][-1]) - float(r['fills_price'][0])
                              for r in results if len(r.get('fills_price', [])) >= 2])
    if not len(price_changes):
        return
    n_up, n_down = (price_changes > 0).sum(), (price_changes < 0).sum()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    _, edges, patches = ax.hist(price_changes, bins=bins, alpha=0.8, edgecolor="white")
    for patch, edge in zip(patches, edges[:-1]):
        if edge < 0:
            patch.set_facecolor('indianred')
        else:
            patch.set_facecolor('mediumseagreen')
    ax.axvline(0, color="grey", lw=1, alpha=0.6)
    ax.axvline(price_changes.mean(), color="crimson", lw=1.4, ls="--",
               label=f"mean = {price_changes.mean():+.4f}")
    ax.axvline(np.median(price_changes), color="darkgreen", lw=1.4, ls=":",
               label=f"median = {np.median(price_changes):+.4f}")
    ax.set(xlabel="Price change per run", ylabel="Count",
           title=f"Price change distribution ({len(price_changes)} runs: \u2191{n_up} \u2193{n_down})")
    ax.grid(alpha=0.25); ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()


def competition_analysis(out_dir):
    """Load competition data from out_dir and plot key metrics.
    Returns (fill_all, state_all, summ_all, Ns, colors).
    """
    setup_dirs = sorted(Path(out_dir).glob("setup_*"))

    def load_parquets(name):
        frames = []
        for setup_dir in setup_dirs:
            if name == "summary":
                summary_path = setup_dir / "summary.parquet"
                if summary_path.is_file():
                    frames.append(pd.read_parquet(summary_path))
            else:
                for run_path in sorted(setup_dir.glob(f"{name}_run*.parquet")):
                    frames.append(pd.read_parquet(run_path))
        if frames:
            return pd.concat(frames, ignore_index=True)
        return pd.DataFrame()

    fill_all  = load_parquets("fill_probe")
    state_all = load_parquets("state")
    summ_all  = load_parquets("summary")

    if summ_all.empty:
        Ns = []
    else:
        Ns = sorted(summ_all["n_agents"].unique())
    fill_all["realized"] = fill_all["filled_qty"] / fill_all["size"].clip(lower=1)
    fill_all["dbucket"]  = fill_all["delta_ticks"].round().clip(lower=0, upper=20)

    cmap = plt.cm.viridis
    colors = {n: cmap(i / max(1, len(Ns) - 1)) for i, n in enumerate(Ns)}

    # Quoted depth vs N (95% CI)
    if not fill_all.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        hs = (fill_all.groupby(["n_agents", "run_id"])["delta_ticks"].mean()
                      .groupby("n_agents").agg(["mean", "std", "count"]))
        se = hs["std"] / np.sqrt(hs["count"].clip(lower=1))
        ax.errorbar(hs.index, hs["mean"], yerr=1.96 * se, marker="o", capsize=4, lw=1.5)
        ax.set_xticks(Ns)
        ax.set(xlabel="number of MMs (N)", ylabel="mean quoted depth \u03b4 (ticks)",
               title="Quoted depth vs competition (95% CI)")
        ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

    # Predicted vs realized fill prob by depth (small multiples per N)
    min_count = 50
    n_col, n_row = 2, int(np.ceil(len(Ns) / 2))
    fig, axes = plt.subplots(n_row, n_col, figsize=(13, 4.2 * n_row),
                             sharex=True, sharey=True, squeeze=False)
    axes = axes.ravel()
    for k, n in enumerate(Ns):
        ax = axes[k]
        depth_stats = (fill_all[fill_all.n_agents == n]
                       .groupby("dbucket").agg(pred=("pred_h", "mean"), real=("realized", "mean"),
                                               cnt=("realized", "size")).reset_index())
        depth_stats = depth_stats[depth_stats.cnt >= min_count].sort_values("dbucket")
        ax.plot(depth_stats.dbucket, depth_stats.real, "-o", color=colors[n], ms=5, lw=2.0, label="realized")
        ax.plot(depth_stats.dbucket, depth_stats.pred, "--s", color="0.35", ms=3, lw=1.4, label="predicted (NN)")
        ax.fill_between(depth_stats.dbucket, depth_stats.pred, depth_stats.real, color=colors[n], alpha=0.12)
        ax.set_title(f"N = {n}"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
        if k % n_col == 0:
            ax.set_ylabel("fill prob per 1s window")
        if k // n_col == n_row - 1:
            ax.set_xlabel("quoted depth (ticks behind mid)")
    for j in range(len(Ns), len(axes)):
        axes[j].axis("off")
    fig.suptitle("Predicted vs realized fill probability by depth", y=1.0, fontsize=13)
    plt.tight_layout(); plt.show()

    # Estimated fill prob vs depth (all N overlaid)
    bucket_stats = (fill_all.groupby(["n_agents", "dbucket"])
                            .agg(pred=("pred_h", "mean"), real=("realized", "mean"),
                                 cnt=("realized", "size")).reset_index())
    fig, ax = plt.subplots(figsize=(8, 5))
    for n in Ns:
        n_stats = bucket_stats[(bucket_stats.n_agents == n) & (bucket_stats.cnt >= 10)]
        n_stats = n_stats.sort_values("dbucket")
        ax.plot(n_stats.dbucket, n_stats.pred, "-o", color=colors[n], ms=4, lw=1.8, label=f"N={n}")
    ax.set(xlabel="quoted depth (ticks behind mid)",
           ylabel="estimated fill prob per 1s window",
           title="Estimated (NN) fill probability vs depth", xlim=(-0.5, 10))
    ax.legend(fontsize=9, title="colour = N"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # PnL vs N
    if not summ_all.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        sim_pnl = (summ_all.groupby(["n_agents", "run_id"])["realized_pnl"].mean()
                           .reset_index())
        pnl_by_n = [sim_pnl.loc[sim_pnl.n_agents == n, "realized_pnl"].to_numpy() for n in Ns]
        ax.boxplot(pnl_by_n, labels=[str(n) for n in Ns], showmeans=True)
        ax.axhline(0, color="black", lw=0.8, ls=":")
        ax.set(xlabel="number of MMs (N)", ylabel="mean realized PnL per sim",
               title="PnL vs competition (sim-level means)")
        ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

    # |inventory| dispersion
    if not state_all.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        rng = np.random.default_rng(0)
        inv_data = []
        for n in Ns:
            abs_inventory = state_all.loc[state_all.n_agents == n, "inventory"].abs().to_numpy()
            if abs_inventory.size > 200_000:
                abs_inventory = rng.choice(abs_inventory, 200_000, replace=False)
            inv_data.append(abs_inventory)
        ax.boxplot(inv_data, labels=[str(n) for n in Ns], showfliers=False)
        ax.set(xlabel="number of MMs (N)", ylabel="|inventory| (shares)",
               title="Inventory dispersion vs competition")
        ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

    if not summ_all.empty:
        print("Per-agent realized PnL by N:")
        print(summ_all.groupby("n_agents")["realized_pnl"].agg(["count", "mean", "median", "std"]))
    near = fill_all[fill_all.dbucket <= 3]
    print("\nMean predicted vs realized fill (depth \u2264 3 ticks) by N:")
    print(near.groupby("n_agents").agg(pred=("pred_h", "mean"), real=("realized", "mean")))

    return fill_all, state_all, summ_all, Ns, colors


def plot_quote_schedule(fill_all, Ns, colors, ylim=(1.0, 3.0)):
    """Quote depth vs inventory for each side and N."""
    if fill_all.empty:
        return
    n_bins, min_per_bin = 15, 50
    quantile_lo, quantile_hi = 0.01, 0.99

    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for side_int, ax, side_label in [(2, ax_a, "Ask"), (1, ax_b, "Bid")]:
        for n in Ns:
            side_fills = fill_all[(fill_all.n_agents == n) & (fill_all.side == side_int)]
            if side_fills.empty:
                continue
            inventory = side_fills["inventory"].to_numpy(dtype=float)
            depth_ticks = side_fills["delta_ticks"].to_numpy(dtype=float)
            edges = np.unique(np.quantile(inventory, np.linspace(quantile_lo, quantile_hi, n_bins + 1)))
            if edges.size < 3:
                continue
            bin_idx = np.clip(np.digitize(inventory, edges[1:-1]), 0, edges.size - 2)
            bin_means_x, bin_means_y = [], []
            for b in range(edges.size - 1):
                in_bin = bin_idx == b
                if int(in_bin.sum()) < min_per_bin:
                    continue
                bin_means_x.append(inventory[in_bin].mean())
                bin_means_y.append(depth_ticks[in_bin].mean())
            if not bin_means_x:
                continue
            ax.plot(np.asarray(bin_means_x), np.asarray(bin_means_y), "-", color=colors[n],
                    lw=1.8, label=f"N={n}")
        ax.axvline(0.0, color="black", lw=0.8, ls=":")
        ax.set(xlabel="inventory level (shares)",
               title=f"Average {side_label} quote depth vs inventory")
        ax.set_ylim(*ylim); ax.grid(alpha=0.3)
        ax.legend(fontsize=9, title="Market makers (N)")
    ax_a.set_ylabel("quoted depth \u03b4 (ticks)")
    plt.tight_layout(); plt.show()

In [ ]:
summaries = {}
for n in (1, 2, 5):
    results = run_setup(n, comp_kwargs)
    summaries[n] = save_setup(results, n, out_dir)
    del results

results_10 = run_setup(10, comp_kwargs)
summaries[10] = save_setup(results_10, 10, out_dir)
del results_10

In [ ]:
rl_base_ckpt = str(resolve_data_path("phantom_models_sim_1M_size10000_logq/fillprob_mlp_best.pt"))
rl_ckpt_dir  = Path(rl_base_ckpt).resolve().parents[1] / "phantom_models_sim_1M_size10000_logq_rl"
rl_ckpt_dir.mkdir(parents=True, exist_ok=True)
rl_latest    = rl_ckpt_dir / "latest.pt"

n_rounds         = 8
sims_per_round   = 12
rl_workers       = 12
t_rl             = 150_000
replay_k         = 4
epochs_per_round = 8
lr               = 1e-4
batch_size       = 65_536
rl_base_seed     = 70_001

rl_kwargs = dict(
    T=t_rl, snapshot_kwargs=snapshot_kwargs,
    erg_params=erg_params_sim, size=size, gamma=gamma,
    solver_tick=sim_tick / 2, max_delta=2.0,
    poisson_tau=1.0, delta_lo=0.0, max_iter=1000, tol=1e-3,
    arrival_mode=arrival_mode, drift_eps=drift_eps,
    requote_cadence=1.0, base_seed=rl_base_seed,
    day_span_s=None, vol_feature_mode="auto",
    agents_affect_kernels=False, agents_affect_mo_sizing=False,
    lo_inside_spread_scale=0.3,
    solver_engine="hull",
    resil_kappa=resil_kappa, resil_tau_s=resil_tau_s,
    resil_phi=resil_phi, resil_flow_tau_s=resil_flow_tau_s,
)

replay = deque(maxlen=replay_k)
rl_history = []
ckpt = rl_base_ckpt

for rnd in range(n_rounds):
    t0 = time.time()

    results = Parallel(n_jobs=min(rl_workers, sims_per_round), verbose=5)(
        delayed(run_fill_rl_sim)(rid, ckpt_path=ckpt, **rl_kwargs)
        for rid in range(sims_per_round)
    )
    X = np.vstack([r["X"] for r in results if r["X"].shape[0] > 0])
    y = np.concatenate([r["y"] for r in results if r["y"].shape[0] > 0])
    pred = np.concatenate([r["pred"] for r in results if r["pred"].shape[0] > 0])

    cal_pred, cal_real, cal_count = calibration_table(pred, y, n_bins=10)
    bce_pre, brier_pre = bce_score(pred, y), brier_score(pred, y)

    replay.append((X, y))
    X_train = np.vstack([batch[0] for batch in replay])
    y_train = np.concatenate([batch[1] for batch in replay])
    round_ckpt = rl_ckpt_dir / f"round_{rnd:02d}.pt"
    info = update_fill_nn(ckpt, X_train, y_train, out_path=str(round_ckpt),
                          epochs=epochs_per_round, lr=lr,
                          batch_size=batch_size, seed=rnd)
    shutil.copyfile(round_ckpt, rl_latest)
    ckpt = str(rl_latest)

    rl_history.append(dict(
        round=rnd, n=int(X.shape[0]), fill_rate=float(y.mean()),
        mean_pred=float(pred.mean()), bce=bce_pre, brier=brier_pre,
        cal_pred=cal_pred, cal_real=cal_real, cal_cnt=cal_count,
        X=X, y=y, pred=pred,
        loss_first=info["loss_first"], loss_last=info["loss_last"],
        round_ckpt=str(round_ckpt)))
    print(f"Round {rnd:02d}: {X.shape[0]:,} rows, "
          f"BCE {info['loss_first']:.5f}\u2192{info['loss_last']:.5f}, "
          f"{(time.time()-t0)/60:.1f} min")

print(f"RL loop done. latest.pt = {rl_latest}")

In [ ]:
fill_all, state_all, summ_all, Ns, colors = competition_analysis(out_dir)

plot_quote_schedule(fill_all, Ns, colors)

assert rl_history, "Run the RL loop cell first."

rounds = [h["round"] for h in rl_history]
colors = cm.viridis(np.linspace(0, 1, len(rl_history)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

# (1) Reliability curve overlaid per round (logged pred vs realized).
ax = axes[0]
hi = max([max(h["cal_pred"].max() if h["cal_pred"].size else 0.0,
              h["cal_real"].max() if h["cal_real"].size else 0.0)
          for h in rl_history] + [1e-3])
ax.plot([0, hi], [0, hi], "k--", lw=1, label="ideal")
for h, c in zip(rl_history, colors):
    if h["cal_pred"].size:
        ax.plot(h["cal_pred"], h["cal_real"], "-o", color=c, ms=3,
                label=f"r{h['round']}")
ax.set_xlabel("predicted fill prob"); ax.set_ylabel("realized fill prob")
ax.set_title("Reliability per round (pre-update)")
ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)

# (2) BCE / Brier vs round.
ax = axes[1]
ax.plot(rounds, [h["bce"] for h in rl_history], "-o", label="BCE")
ax.plot(rounds, [h["brier"] for h in rl_history], "-s", label="Brier")
ax.set_xlabel("round"); ax.set_ylabel("loss (pre-update, logged net)")
ax.set_title("Calibration loss vs round"); ax.grid(alpha=0.3); ax.legend()

# (3) Predicted vs realized fill-by-depth (ticks) overlaid across rounds.
ax = axes[2]
base_bundle = mm_backtest_parallel._load_nn_bundle(rl_base_ckpt)
feat_mean, feat_std, n_feat = base_bundle["feat_mean"], base_bundle["feat_std"], base_bundle["n_feat"]
edges = np.arange(0, 11, 1)                       # depth buckets in ticks
centers = 0.5 * (edges[:-1] + edges[1:])
for h, c in zip(rl_history, colors):
    depth_ticks = delta_ticks_from_X(h["X"], feat_mean, feat_std, n_feat, sim_tick)
    idx = np.clip(np.digitize(depth_ticks, edges) - 1, 0, len(centers) - 1)
    real_by = []
    pred_by = []
    for b in range(len(centers)):
        in_bucket = idx == b
        if in_bucket.any():
            real_by.append(h["y"][in_bucket].mean())
            pred_by.append(h["pred"][in_bucket].mean())
        else:
            real_by.append(np.nan)
            pred_by.append(np.nan)
    ax.plot(centers, real_by, "-o", color=c, ms=3, label=f"r{h['round']} real")
    ax.plot(centers, pred_by, "--", color=c, lw=1)
ax.set_xlabel("quoted depth (ticks behind mid)")
ax.set_ylabel("fill prob per 1s window")
ax.set_title("Fill-by-depth: realized (solid) vs predicted (dashed)")
ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)

plt.tight_layout(); plt.show()

print("Per-round summary:")
print(pd.DataFrame([{k: h[k] for k in
                     ("round", "n", "fill_rate", "mean_pred", "bce", "brier",
                      "loss_first", "loss_last")} for h in rl_history])
      .to_string(index=False))